In [4]:
using Luxor
using Colors
using Plots
using IterTools
using DataFrames
using OpenStreetMapX
using LightOSM
using KernelDensity
using Downloads
include("../kernel_density.jl")
include("../distance.jl")
include("../prepare_data.jl")
include("../analyse.jl")
include("../plots.jl")
include("/home/adamkas/Julia/map_analyses/OSMXGraph/src/OSMXGraph.jl")
using .OSMXGraph

In [5]:
percentiles = Dict()

Dict{Any, Any}()

In [111]:
#cities1 = ["Kielce","Kraków","Warszawa","Brno","Vienna","Toronto"]
#cities2 = ["Trieste","Vilnius","Zagreb","Sofia","Sewilla","Seattle"]
#cities3 = ["San Diego","San Francisko","Sacramento","Rome", "Quebec City"]
#cities4 = ["Prague","Poznan","Pittsburgh","Ottawa","Oslo", "New York"]
#cities5 = ["Naples", "Munich", "Montreal", "Milan", "Miami", "Madrid"]
#cities6 = ["Lyon", "Los Angeles", "Lisbon","Istanbul","Hague"]
#cities7 = ["Hague","Dublin","Detroit","Denver","Dallas","Copenhagen","Chicago"] "Copenhagen"
#cities8 = ["Chicago","Budapest","Bucharest","Bratislava","Boston","Belgrad","Barcelona"]
#cities9 = ["Barcelona","Baltimore","Austin", "Atlanta", "Athens","Ankara","Amsterdam"]
cities = ["Cleveland","Tampa","Milwaukee","Memphis",
          "Columbus","Boise","Portland","New Orleans","Kansas City"]
#cities= ["Houston","Phoenix","Florence","Calgary",
#        "Vancouver","Minneapolis","Katowice"], "Helsinki","Rotterdam","Cologne","Manchester","Marseille",
            #"Nashville","Birmingham","Antwerp","Bilbao","Lille","Winnipeg","Halifax","Regina","Whitehorse"

9-element Vector{String}:
 "Cleveland"
 "Tampa"
 "Milwaukee"
 "Memphis"
 "Columbus"
 "Boise"
 "Portland"
 "New Orleans"
 "Kansas City"

In [112]:
#cities_list = [cities1, cities2, cities3, cities4, 
#                cities5, cities6, cities7, cities8, cities9]

cities_list = [cities]

1-element Vector{Vector{String}}:
 ["Cleveland", "Tampa", "Milwaukee", "Memphis", "Columbus", "Boise", "Portland", "New Orleans", "Kansas City"]

In [113]:
for cities in cities_list
    for ct in cities
        data = "../data"
        city = ct
        admin_level = "asd6"
        search_area = 1000
        attr = :education
        wilderness_distance = 300
        shape = "rectangle"
        calculate_percent = true
        num_of_points = 360
        distance_sectors = 200.0
        scrape_config = "../poi_config_test.csv"
        city_sector = prepare_city_map(
                    city, #city_name
                    admin_level, #admin_level
                    search_area, #search_area
                    wilderness_distance, #wilderness_distance
                    shape; #shape
                    calculate_percent = true,
                    num_of_points = num_of_points,
                    scrape_config = scrape_config,
                    dir=data)

        dir_in = "../data"
        road_types = ["motorway", "trunk", "primary", "secondary", 
                    "tertiary", "residential", "service", "living_street", 
                    "motorway_link", "trunk_link", "primary_link", "secondary_link", 
                    "tertiary_link"] 
        osm_file = "$city.osm"
        #graph_file_name = "Warszawa_graph.csv"
        #node_file_name = "Warszawa_nodes.json"
        dir_in=dir_in
        parsed = OpenStreetMapX.parseOSM(string(dir_in,"/",osm_file))
        ways = parsed.ways
        filtered_ways = OSMXGraph.filter_ways(ways,road_types)
        roads_all, road_tags, nds_all = OSMXGraph.find_all_points(filtered_ways, parsed)
        edges = OSMXGraph.ways_to_edges(roads_all,road_tags,parsed,nds_all)
        df = OSMXGraph.edges_to_df(edges)
        sparse_index = OSMXGraph.create_sparse_index(df.from_id,df.to_id,df.id)
        warsaw_center = city_sector[2]
        nodes = df[:,:from_LLA]
        rslts, center_kde = kernel_density_roads(city_sector,nodes)
        rs = vec(rslts)
        percentiles[city] = mean(center_kde .>= rs)
    end
end

┌ Info: Saved map data to cache ../data/Cleveland.osm.cache
└ @ OpenStreetMapX /home/adamkas/.julia/packages/OpenStreetMapX/gCd33/src/parseMap.jl:110
┌ Info: Saved map data to cache ../data/Tampa.osm.cache
└ @ OpenStreetMapX /home/adamkas/.julia/packages/OpenStreetMapX/gCd33/src/parseMap.jl:110
┌ Info: Saved map data to cache ../data/Milwaukee.osm.cache
└ @ OpenStreetMapX /home/adamkas/.julia/packages/OpenStreetMapX/gCd33/src/parseMap.jl:110
┌ Info: Saved map data to cache ../data/Memphis.osm.cache
└ @ OpenStreetMapX /home/adamkas/.julia/packages/OpenStreetMapX/gCd33/src/parseMap.jl:110
┌ Info: Saved map data to cache ../data/Columbus.osm.cache
└ @ OpenStreetMapX /home/adamkas/.julia/packages/OpenStreetMapX/gCd33/src/parseMap.jl:110
┌ Info: Saved map data to cache ../data/Boise.osm.cache
└ @ OpenStreetMapX /home/adamkas/.julia/packages/OpenStreetMapX/gCd33/src/parseMap.jl:110
┌ Info: Saved map data to cache ../data/Portland.osm.cache
└ @ OpenStreetMapX /home/adamkas/.julia/packages/Ope

In [12]:
#plot_heatmap(city_sector[1], 
#            rslts, 
#            city_sector[5], 
#            :density, 
#            "Warszawa",
#            search_area,
#            wilderness_distance,
#            add_center=true)

In [116]:
percentiles

Dict{Any, Any} with 44 entries:
  "Manchester"  => 0.955734
  "Kansas City" => 0.999208
  "Berlin"      => 0.962873
  "Katowice"    => 0.982313
  "Porto"       => 0.669781
  "Houston"     => 0.996303
  "Leeds"       => 0.997999
  "Calgary"     => 0.99042
  "Paris"       => 0.923928
  "Winnipeg"    => 0.999928
  "Columbus"    => 0.970041
  "Frankfurt"   => 0.986919
  "Lille"       => 0.706619
  "Cologne"     => 0.999273
  "Hamilton"    => 0.999879
  "Tampa"       => 0.999602
  "Vancouver"   => 0.974557
  "Whitehorse"  => 0.999802
  "Edmonton"    => 0.981126
  ⋮             => ⋮

In [117]:
cities_df = collect(keys(percentiles))
values_df = collect(values(percentiles))

44-element Vector{Any}:
 0.9557344064386318
 0.999208089668616
 0.9628726287262873
 0.9823134398940698
 0.669780511885775
 0.9963030303030304
 0.9979990583804144
 0.990419997462251
 0.9239278210357187
 0.9999279953917051
 ⋮
 0.9298174986580784
 0.9348086124401914
 0.9980031379261161
 0.9988851727982163
 0.9469615051903114
 0.8241807475678443
 0.6415787791626718
 0.9869998401619692
 0.9258548790658883

In [118]:
df = DataFrame(City = cities_df, Value = values_df)

Row,City,Value
,Any,Any
1,Manchester,0.955734
2,Kansas City,0.999208
3,Berlin,0.962873
4,Katowice,0.982313
5,Porto,0.669781
6,Houston,0.996303
7,Leeds,0.997999
8,Calgary,0.99042
9,Paris,0.923928


In [129]:
df[df.City .== "Kansas City",:]

Row,City,Value
,Any,Any
1,Kansas City,0.999208


In [ ]:
CSV.write("KDE_results_2.csv",df)

"KDE_results.csv"